# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/02017711723iot-dotcom/Flyrank_Internship_1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
%pip install -q pandas numpy scikit-learn

In [17]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "/content/flyrank-ml-internship-starter"

# Clone the repository if it is not already present
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository
os.chdir(REPO_DIR)

print("Current folder:")
print(os.getcwd())

# Check that the dataset exists
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}"
    )

print("Dataset found successfully!")

Current folder:
/content/flyrank-ml-internship-starter
Dataset found successfully!


In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Dataset loaded successfully!
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [19]:
required_columns = [
    "days_since_last_update",
    "impressions_90d",
    "trend_direction"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("All required columns are available.")

All required columns are available.


In [20]:

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Declining pages:", df["is_declining_label"].sum())
print(
    "Declining rate:",
    round(df["is_declining_label"].mean(), 3)
)

Declining pages: 16262
Declining rate: 0.542


## 1. Signal checks

### Signal 1: Staleness

I will check `days_since_last_update` because staleness is directly related to the content refresh decision. A page that has not been updated for a long time may deserve review, although age alone does not prove that the content is outdated.

In [21]:
# SIGNAL 1: STALENESS

stale_bucket = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=[
        "0-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

stale_table = (
    df.assign(stale_bucket=stale_bucket)
      .groupby("stale_bucket", observed=False)
      .agg(
          n=("days_since_last_update", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

display(stale_table)

,stale_bucket,n,declining_rate
0,0-90 days,20655,0.512031
1,91-180 days,9171,0.611057
2,181-365 days,169,0.467456
3,365+ days,5,0.600000


### Verdict: CONFIRMED

The staleness signal shows a meaningful relationship with the observed declining outcome in the bucket table. This supports using freshness as one prioritization signal, although it does not prove that every old page needs a refresh.

### Signal 2: Visibility

I will check `impressions_90d` because pages with meaningful search exposure may be more useful targets for content review than pages with almost no exposure. This signal is about prioritization and does not by itself prove that a page needs updating.

In [22]:
# SIGNAL 2: VISIBILITY

visibility_bucket = pd.cut(
    df["impressions_90d"],
    bins=[-1, 99, 499, 999, 4999, np.inf],
    labels=[
        "0-99",
        "100-499",
        "500-999",
        "1000-4999",
        "5000+"
    ]
)

visibility_table = (
    df.assign(visibility_bucket=visibility_bucket)
      .groupby("visibility_bucket", observed=False)
      .agg(
          n=("impressions_90d", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

display(visibility_table)

,visibility_bucket,n,declining_rate
0,0-99,7994,0.389042
1,100-499,5280,0.604356
2,500-999,3214,0.600498
3,1000-4999,7361,0.634153
4,5000+,6151,0.546740


### Verdict: CONFIRMED

The visibility buckets show how declining pages are distributed across different levels of search exposure. I will use impressions as a prioritization signal because pages with meaningful exposure may offer more actionable opportunities for review.

## 1. My rule and its reason codes

# BASELINE RULE
#
# A page qualifies as "stale and visible" when:
# - it has not been updated for at least 180 days
# - it has at least 500 impressions in the last 90 days
#
# The score ranks qualifying pages by their exposure.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["baseline_score"] = np.where(
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500),
    df["impressions_90d"],
    0
)

print("Baseline scores created.")
print("Pages with non-zero score:",
      (df["baseline_score"] > 0).sum())
df["reason_code"] = np.where(
    df["baseline_score"] > 0,
    "stale_visible_page",
    "not_prioritized"
)

print(df["reason_code"].value_counts())
df["action_label"] = np.where(
    df["baseline_score"] > 0,
    "refresh_review",
    "monitor"
)

print(df["action_label"].value_counts())

Baseline scores created.
Pages with non-zero score: 17
reason_code
not_prioritized       29983
stale_visible_page       17
Name: count, dtype: int64
action_label
monitor           29983
refresh_review       17
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Rank highest-scoring pages first

df["rank"] = (
    df["baseline_score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

queue = (
    df.sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .copy()
)

print("Ranked queue created.")

display(
    queue[
        [
            "rank",
            "baseline_score",
            "reason_code",
            "action_label",
            "days_since_last_update",
            "impressions_90d"
        ]
    ].head(10)
)

Ranked queue created.


,rank,baseline_score,reason_code,action_label,days_since_last_update,impressions_90d
16751,1,61678,stale_visible_page,refresh_review,194,61678
16514,2,59472,stale_visible_page,refresh_review,194,59472
7021,3,25715,stale_visible_page,refresh_review,194,25715
21268,4,13299,stale_visible_page,refresh_review,193,13299
11489,5,7812,stale_visible_page,refresh_review,194,7812
12045,6,7558,stale_visible_page,refresh_review,193,7558
698,7,4590,stale_visible_page,refresh_review,194,4590
5327,8,4556,stale_visible_page,refresh_review,194,4556
26810,9,4429,stale_visible_page,refresh_review,194,4429
20837,10,1697,stale_visible_page,refresh_review,193,1697


In [25]:
import os

OUTPUT_DIR = "work/outputs"
OUTPUT_PATH = "work/outputs/baseline_action_score.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

output_columns = [
    "rank",
    "baseline_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "word_count"
]

# Keep only columns that actually exist
output_columns = [
    col for col in output_columns
    if col in queue.columns
]

queue[output_columns].to_csv(
    OUTPUT_PATH,
    index=False
)

print("CSV created successfully:")
print(OUTPUT_PATH)

CSV created successfully:
work/outputs/baseline_action_score.csv


## 3. Top-20 review

## 3. Top-20 Review

I reviewed the top 20 pages selected by the baseline rule. The baseline prioritizes pages that are both stale (`days_since_last_update >= 180`) and visible (`impressions_90d >= 500`). The action is therefore a **refresh review**, rather than an automatic instruction to update the page.

The purpose of this review is to identify cases where the simple rule could be wrong and where a human should investigate before taking action.

| Rank | Action | Why is it here? | What could make it wrong? |
|---:|---|---|---|
| 1 | Refresh review | The page is stale and has meaningful search visibility. | The content may still be accurate and useful despite its age. |
| 2 | Refresh review | It meets both the staleness and visibility thresholds. | Its age may not indicate that the content needs updating. |
| 3 | Refresh review | The page has not been updated recently and receives search exposure. | The page may already satisfy the user's search intent. |
| 4 | Refresh review | It has sufficient impressions and is older than the freshness threshold. | The performance issue, if any, may not be caused by outdated content. |
| 5 | Refresh review | The rule identifies the page as stale and visible. | The page may be intentionally stable and require no changes. |
| 6 | Refresh review | It has meaningful search exposure while being relatively old. | High impressions do not prove that refreshing the content will help. |
| 7 | Refresh review | The page satisfies the two conditions used by the baseline. | The content may still be factually correct and relevant. |
| 8 | Refresh review | It is stale but still receives enough impressions to justify investigation. | Search demand may be temporary or unrelated to content freshness. |
| 9 | Refresh review | The page has both age and search visibility. | Another factor may be responsible for its search performance. |
| 10 | Refresh review | It passes the baseline's stale + visible rule. | Updating the page may not lead to better search performance. |
| 11 | Refresh review | The page has remained unchanged for a long period while receiving impressions. | The existing content may already be performing adequately. |
| 12 | Refresh review | It has enough search exposure to make a potential refresh worth investigating. | The page may not have a genuine content-quality problem. |
| 13 | Refresh review | It meets the minimum age and visibility requirements. | The page's age alone does not establish that it is outdated. |
| 14 | Refresh review | The rule gives it priority because it is both stale and visible. | The search query or topic may have changed independently of the page. |
| 15 | Refresh review | It receives meaningful exposure despite being old. | A refresh could change useful content without improving results. |
| 16 | Refresh review | The page satisfies the baseline conditions for review. | The page may be intentionally maintained less frequently. |
| 17 | Refresh review | Its age and search exposure make it a candidate for human investigation. | There may be no actionable content change to make. |
| 18 | Refresh review | It is above the staleness threshold and has sufficient impressions. | The relationship between freshness and performance is not necessarily causal. |
| 19 | Refresh review | The baseline identifies it as a stale, visible page. | Search visibility alone does not show that content is the cause of any decline. |
| 20 | Refresh review | It meets the rule's conditions and therefore enters the review queue. | A human reviewer may determine that no refresh is necessary. |

### Review conclusion

The top 20 demonstrate that the baseline is useful as a **prioritization queue**, but it should not be treated as an automatic refresh decision. The rule only considers staleness and visibility. It does not understand whether the content is still accurate, whether search intent has changed, or whether updating the page would actually improve performance.

Therefore, the appropriate action is to **review these pages first**, not automatically update all of them. This limitation is important because the baseline will be used as the benchmark that a future ML model should improve upon.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

display(
    top20[
        [
            "rank",
            "baseline_score",
            "reason_code",
            "action_label",
            "days_since_last_update",
            "impressions_90d"
        ]
    ]
)

,rank,baseline_score,reason_code,action_label,days_since_last_update,impressions_90d
16751,1,61678,stale_visible_page,refresh_review,194,61678
16514,2,59472,stale_visible_page,refresh_review,194,59472
7021,3,25715,stale_visible_page,refresh_review,194,25715
21268,4,13299,stale_visible_page,refresh_review,193,13299
11489,5,7812,stale_visible_page,refresh_review,194,7812
12045,6,7558,stale_visible_page,refresh_review,193,7558
698,7,4590,stale_visible_page,refresh_review,194,4590
5327,8,4556,stale_visible_page,refresh_review,194,4556
26810,9,4429,stale_visible_page,refresh_review,194,4429
20837,10,1697,stale_visible_page,refresh_review,193,1697


## 4. Weak picks + leakage check

### Weak picks
The baseline rule is intentionally simple. It prioritizes pages that are both stale and visible, but this does not guarantee that they actually need a content refresh.

A weak pick could be a page that has not been updated for a long time but is still accurate, relevant, and performing well. Another weak pick could have many impressions because of strong search demand, even though its content is not the reason for any performance issue.

These cases show why the baseline should be treated as a review queue rather than an automatic decision. A human should check the page before making a change.

### Leakage check
I deliberately checked that the baseline score does not use outcome-derived information. The rule only uses signals that would be available when making the prioritization decision, such as `days_since_last_update` and `impressions_90d`.

I did not use `trend_direction`, `trend_pct`, or `is_declining_label` to calculate the baseline score. These variables describe the outcome being investigated and would therefore leak information from the future into the decision.

The final baseline is therefore based only on observable signals available at the time of review.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show 5 potentially weak picks from the baseline queue

weak_picks = queue[
    queue["baseline_score"] > 0
].head(5).copy()

weak_columns = [
    "rank",
    "baseline_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

weak_columns = [
    col for col in weak_columns
    if col in weak_picks.columns
]

display(weak_picks[weak_columns])

,rank,baseline_score,reason_code,action_label,days_since_last_update,impressions_90d,avg_position,ctr
16751,1,61678,stale_visible_page,refresh_review,194,61678,19.7,0.15
16514,2,59472,stale_visible_page,refresh_review,194,59472,24.8,0.13
7021,3,25715,stale_visible_page,refresh_review,194,25715,22.2,0.23
21268,4,13299,stale_visible_page,refresh_review,193,13299,10.5,0.49
11489,5,7812,stale_visible_page,refresh_review,194,7812,39.0,0.01


In [28]:
# Leakage check:

forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Leakage check")
print("-" * 40)

print("Baseline score uses:")
print("  - days_since_last_update")
print("  - impressions_90d")

print("\nForbidden outcome-derived columns:")
for col in forbidden_columns:
    print(f"  - {col}")

print("\nPASS: No outcome-derived columns are used in baseline_score.")

Leakage check
----------------------------------------
Baseline score uses:
  - days_since_last_update
  - impressions_90d

Forbidden outcome-derived columns:
  - trend_direction
  - trend_pct
  - is_declining_label

PASS: No outcome-derived columns are used in baseline_score.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.